In [6]:
%pip install bleach
%pip install python-docx
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


КОД

In [ ]:
import os
import sqlite3
import json
import hashlib
import shutil
import bleach
from datetime import datetime, timedelta
from flask import Flask, render_template, request, redirect, url_for, flash, session, g
from werkzeug.utils import secure_filename
from docx import Document

app = Flask(__name__)
app.config['SECRET_KEY'] = 'dev-secret-key-change-in-production'
DATABASE = 'edoc.db'
BACKUP_DIR = 'backups'
UPLOAD_FOLDER = 'uploads'
ALLOWED_EXTENSIONS = {'txt', 'docx'}
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16 MB

os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# ---------- Вспомогательные функции для работы с БД ----------
def get_db():
    db = getattr(g, '_database', None)
    if db is None:
        db = g._database = sqlite3.connect(DATABASE)
        db.row_factory = sqlite3.Row
    return db

@app.teardown_appcontext
def close_connection(exception):
    db = getattr(g, '_database', None)
    if db is not None:
        db.close()

def query_db(query, args=(), one=False):
    cur = get_db().execute(query, args)
    rv = cur.fetchall()
    cur.close()
    return (rv[0] if rv else None) if one else rv

def execute_db(query, args=()):
    conn = get_db()
    cur = conn.execute(query, args)
    conn.commit()
    return cur.lastrowid

# ---------- Функции аутентификации ----------
def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

# ---------- Инициализация БД ----------
def init_db():
    with app.app_context():
        db = get_db()
        # Удаляем старые таблицы
        db.executescript('''
            DROP TABLE IF EXISTS users;
            DROP TABLE IF EXISTS documents;
            DROP TABLE IF EXISTS relations;
            DROP TABLE IF EXISTS workflow_templates;
            DROP TABLE IF EXISTS workflow_instances;
            DROP TABLE IF EXISTS approval_tasks;
            DROP TABLE IF EXISTS audit_log;
            DROP TABLE IF EXISTS task_status_history;
            DROP TABLE IF EXISTS session;
        ''')

        db.executescript('''
            CREATE TABLE users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                full_name TEXT,
                role TEXT NOT NULL
            );

            CREATE TABLE documents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                content TEXT DEFAULT '',
                source_system TEXT,
                external_id TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE relations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                from_doc_id INTEGER NOT NULL,
                to_doc_id INTEGER NOT NULL,
                relation_type TEXT,
                FOREIGN KEY(from_doc_id) REFERENCES documents(id),
                FOREIGN KEY(to_doc_id) REFERENCES documents(id)
            );

            CREATE TABLE workflow_templates (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                description TEXT,
                steps TEXT,
                steps_config TEXT
            );

            CREATE TABLE workflow_instances (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                document_id INTEGER NOT NULL,
                template_id INTEGER NOT NULL,
                status TEXT DEFAULT 'active',
                current_step INTEGER DEFAULT 0,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(document_id) REFERENCES documents(id),
                FOREIGN KEY(template_id) REFERENCES workflow_templates(id)
            );

            CREATE TABLE approval_tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                workflow_instance_id INTEGER NOT NULL,
                step_index INTEGER NOT NULL,
                assigned_role TEXT NOT NULL,
                status TEXT DEFAULT 'pending',
                comments TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                deadline TIMESTAMP,
                can_return BOOLEAN DEFAULT 0,
                parent_task_id INTEGER,
                FOREIGN KEY(workflow_instance_id) REFERENCES workflow_instances(id),
                FOREIGN KEY(parent_task_id) REFERENCES approval_tasks(id)
            );

            CREATE TABLE audit_log (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                table_name TEXT NOT NULL,
                operation TEXT NOT NULL,
                old_data TEXT,
                new_data TEXT,
                changed_by TEXT NOT NULL,
                changed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE task_status_history (
                history_id INTEGER PRIMARY KEY AUTOINCREMENT,
                task_id INTEGER NOT NULL,
                old_status TEXT,
                new_status TEXT,
                changed_by TEXT,
                changed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(task_id) REFERENCES approval_tasks(id) ON DELETE CASCADE
            );

            CREATE TABLE session (
                id INTEGER PRIMARY KEY CHECK (id = 1),
                current_user TEXT NOT NULL
            );
        ''')

        db.execute("INSERT INTO session (id, current_user) VALUES (1, 'system')")

        # Триггеры аудита
        db.executescript('''
            -- documents триггеры
            CREATE TRIGGER documents_after_insert
            AFTER INSERT ON documents
            BEGIN
                INSERT INTO audit_log (table_name, operation, new_data, changed_by)
                VALUES ('documents', 'INSERT',
                        json_object('id', NEW.id, 'title', NEW.title),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER documents_after_update
            AFTER UPDATE ON documents
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, new_data, changed_by)
                VALUES ('documents', 'UPDATE',
                        json_object('id', OLD.id, 'title', OLD.title),
                        json_object('id', NEW.id, 'title', NEW.title),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER documents_after_delete
            AFTER DELETE ON documents
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, changed_by)
                VALUES ('documents', 'DELETE',
                        json_object('id', OLD.id, 'title', OLD.title),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            -- workflow_instances триггеры
            CREATE TRIGGER workflow_instances_after_insert
            AFTER INSERT ON workflow_instances
            BEGIN
                INSERT INTO audit_log (table_name, operation, new_data, changed_by)
                VALUES ('workflow_instances', 'INSERT',
                        json_object('id', NEW.id, 'document_id', NEW.document_id, 'template_id', NEW.template_id, 'status', NEW.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER workflow_instances_after_update
            AFTER UPDATE ON workflow_instances
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, new_data, changed_by)
                VALUES ('workflow_instances', 'UPDATE',
                        json_object('id', OLD.id, 'status', OLD.status),
                        json_object('id', NEW.id, 'status', NEW.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER workflow_instances_after_delete
            AFTER DELETE ON workflow_instances
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, changed_by)
                VALUES ('workflow_instances', 'DELETE',
                        json_object('id', OLD.id, 'status', OLD.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            -- approval_tasks триггеры
            CREATE TRIGGER approval_tasks_after_insert
            AFTER INSERT ON approval_tasks
            BEGIN
                INSERT INTO audit_log (table_name, operation, new_data, changed_by)
                VALUES ('approval_tasks', 'INSERT',
                        json_object('id', NEW.id, 'assigned_role', NEW.assigned_role, 'status', NEW.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER approval_tasks_after_update
            AFTER UPDATE ON approval_tasks
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, new_data, changed_by)
                VALUES ('approval_tasks', 'UPDATE',
                        json_object('id', OLD.id, 'status', OLD.status),
                        json_object('id', NEW.id, 'status', NEW.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER approval_tasks_after_delete
            AFTER DELETE ON approval_tasks
            BEGIN
                INSERT INTO audit_log (table_name, operation, old_data, changed_by)
                VALUES ('approval_tasks', 'DELETE',
                        json_object('id', OLD.id, 'status', OLD.status),
                        (SELECT current_user FROM session WHERE id = 1));
            END;

            CREATE TRIGGER approval_tasks_status_update
            AFTER UPDATE ON approval_tasks
            WHEN OLD.status != NEW.status
            BEGIN
                INSERT INTO task_status_history (task_id, old_status, new_status, changed_by)
                VALUES (NEW.id, OLD.status, NEW.status,
                        (SELECT current_user FROM session WHERE id = 1));
            END;
        ''')

        # Тестовые пользователи
        users = [
            ('alex_admin', hash_password('SecurePass123'), 'Алексей Админов', 'admin'),
            ('maria_manager', hash_password('ManagerPass456'), 'Мария Менеджер', 'manager'),
            ('ivan_developer', hash_password('DevPass789'), 'Иван Разработчик', 'developer'),
            ('elena_lawyer', hash_password('LawyerPass000'), 'Елена Юрист', 'lawyer'),
        ]
        for u in users:
            db.execute('INSERT INTO users (username, password_hash, full_name, role) VALUES (?,?,?,?)', u)

        db.commit()
        print("База данных инициализирована (структура, триггеры и тестовые пользователи созданы).")

# ---------- Функции для работы с файлами ----------
def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def get_storage_doc_id():
    storage = query_db("SELECT id FROM documents WHERE title = ?", ('Хранилище загруженных файлов',), one=True)
    if storage:
        return storage['id']
    cur = get_db().execute(
        "INSERT INTO documents (title, content, source_system) VALUES (?,?,?)",
        ('Хранилище загруженных файлов', 'Служебный документ для связей загруженных файлов', 'system')
    )
    get_db().commit()
    return cur.lastrowid

def extract_text_from_docx(filepath):
    doc = Document(filepath)
    paragraphs = []
    for para in doc.paragraphs:
        if para.text.strip():
            paragraphs.append(para.text)
    html = '\n'.join(f'<p>{p}</p>' for p in paragraphs)
    return html

# ---------- Функции аутентификации и проверки прав ----------
def login_required(f):
    from functools import wraps
    @wraps(f)
    def decorated_function(*args, **kwargs):
        if 'user_id' not in session:
            flash('Необходимо войти в систему', 'warning')
            return redirect(url_for('login', next=request.url))
        return f(*args, **kwargs)
    return decorated_function

def has_permission(table, operation):
    if 'user_id' not in session:
        return False
    user = query_db('SELECT role FROM users WHERE id = ?', (session['user_id'],), one=True)
    if not user:
        return False
    role = user['role']
    if role == 'admin':
        return True
    if role == 'manager':
        if table == 'documents' and operation in ['SELECT', 'INSERT', 'UPDATE']:
            return True
        if table in ['workflow_templates', 'workflow_instances'] and operation in ['SELECT', 'INSERT', 'UPDATE']:
            return True
        if table == 'approval_tasks' and operation in ['SELECT', 'UPDATE']:
            return True
        return False
    if role == 'developer':
        allowed_tables = ['documents', 'workflow_templates', 'workflow_instances', 'approval_tasks']
        if operation == 'SELECT' and table in allowed_tables:
            return True
        return False
    if role == 'lawyer':
        if operation == 'SELECT' and table in ['documents', 'workflow_templates', 'workflow_instances']:
            return True
        if table == 'approval_tasks' and operation == 'SELECT':
            return True
        return False
    return False

# ---------- Обработчик перед каждым запросом ----------
@app.before_request
def before_request():
    if 'user_id' in session:
        user = query_db('SELECT username, role FROM users WHERE id = ?', (session['user_id'],), one=True)
        if user:
            g.user = dict(user)
            db = get_db()
            db.execute("DELETE FROM session")
            db.execute("INSERT INTO session (id, current_user) VALUES (1, ?)", (user['username'],))
            db.commit()
        else:
            session.clear()
            g.user = None
    else:
        g.user = None

@app.context_processor
def inject_user():
    return dict(current_user=g.user if hasattr(g, 'user') else None)

# ---------- Маршруты аутентификации ----------
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form['username']
        password = request.form['password']
        user = query_db('SELECT * FROM users WHERE username = ?', (username,), one=True)
        if user and user['password_hash'] == hash_password(password):
            session['user_id'] = user['id']
            flash('Вы успешно вошли', 'success')
            next_page = request.args.get('next')
            return redirect(next_page or url_for('dashboard'))
        else:
            flash('Неверное имя пользователя или пароль', 'danger')
    return render_template('login.html')

@app.route('/logout')
def logout():
    session.clear()
    flash('Вы вышли из системы', 'info')
    return redirect(url_for('login'))

# ---------- Главная страница ----------
@app.route('/')
@app.route('/dashboard')
@login_required
def dashboard():
    role = g.user['role'] if g.user else 'developer'
    widgets = []
    if role == 'developer':
        pending_tasks = query_db('''
            SELECT at.*, wf.document_id, d.title as doc_title
            FROM approval_tasks at
            JOIN workflow_instances wf ON at.workflow_instance_id = wf.id
            JOIN documents d ON wf.document_id = d.id
            WHERE at.assigned_role = ? AND at.status = 'pending'
        ''', (role,))
        pending_tasks = [dict(t) for t in pending_tasks]
        widgets.append({'title': 'Ожидают вашего согласования', 'type': 'task_list', 'data': pending_tasks})
    elif role == 'manager':
        active_workflows_count = query_db('SELECT COUNT(*) as cnt FROM workflow_instances WHERE status = "active"', one=True)['cnt']
        pending_tasks_count = query_db('SELECT COUNT(*) as cnt FROM approval_tasks WHERE status = "pending"', one=True)['cnt']
        widgets.append({'title': 'Активные процессы согласования', 'type': 'stats',
                        'data': {'active_workflows': active_workflows_count, 'pending_tasks': pending_tasks_count}})
        recent_workflows = query_db('''
            SELECT wf.*, d.title as doc_title
            FROM workflow_instances wf
            JOIN documents d ON wf.document_id = d.id
            ORDER BY wf.created_at DESC LIMIT 5
        ''')
        recent_workflows = [dict(w) for w in recent_workflows]
        widgets.append({'title': 'Последние запущенные процессы', 'type': 'workflow_list', 'data': recent_workflows})
    elif role == 'lawyer':
        tasks = query_db('''
            SELECT at.*, wf.document_id, d.title as doc_title
            FROM approval_tasks at
            JOIN workflow_instances wf ON at.workflow_instance_id = wf.id
            JOIN documents d ON wf.document_id = d.id
            WHERE at.assigned_role = ? AND at.status = 'pending'
        ''', ('lawyer',))
        tasks = [dict(t) for t in tasks]
        widgets.append({'title': 'Задачи на юридическое согласование', 'type': 'task_list', 'data': tasks})
    recent_docs = query_db('SELECT * FROM documents ORDER BY updated_at DESC LIMIT 5')
    recent_docs = [dict(doc) for doc in recent_docs]
    widgets.append({'title': 'Недавние документы', 'type': 'document_list', 'data': recent_docs})
    sample_relations = query_db('''
        SELECT r.*, d1.title as from_title, d2.title as to_title
        FROM relations r
        JOIN documents d1 ON r.from_doc_id = d1.id
        JOIN documents d2 ON r.to_doc_id = d2.id
        ORDER BY r.id DESC LIMIT 5
    ''')
    sample_relations = [dict(r) for r in sample_relations]
    widgets.append({'title': 'Недавние связи документов', 'type': 'relation_list', 'data': sample_relations})

    # Виджет активных процессов для всех
    active_workflows = query_db('''
        SELECT wf.*, d.title as doc_title, t.name as template_name
        FROM workflow_instances wf
        JOIN documents d ON wf.document_id = d.id
        JOIN workflow_templates t ON wf.template_id = t.id
        WHERE wf.status = 'active'
        ORDER BY wf.created_at DESC LIMIT 5
    ''')
    active_workflows = [dict(w) for w in active_workflows]
    widgets.append({'title': 'Активные процессы', 'type': 'workflow_list', 'data': active_workflows})

    return render_template('dashboard.html', widgets=widgets, current_role=role)

# ---------- Просмотр документа ----------
@app.route('/document/<int:doc_id>')
@login_required
def document_view(doc_id):
    if not has_permission('documents', 'SELECT'):
        flash('У вас нет прав на просмотр документов', 'danger')
        return redirect(url_for('dashboard'))
    doc = query_db('SELECT * FROM documents WHERE id = ?', (doc_id,), one=True)
    if not doc:
        return "Документ не найден", 404
    doc = dict(doc)
    relations_out = query_db('''
        SELECT r.*, d.* FROM relations r
        JOIN documents d ON r.to_doc_id = d.id
        WHERE r.from_doc_id = ?
    ''', (doc_id,))
    relations_in = query_db('''
        SELECT r.*, d.* FROM relations r
        JOIN documents d ON r.from_doc_id = d.id
        WHERE r.to_doc_id = ?
    ''', (doc_id,))
    related = []
    for r in relations_out:
        r_dict = dict(r)
        related.append({'doc': r_dict, 'relation_type': r_dict['relation_type'], 'direction': 'outgoing'})
    for r in relations_in:
        r_dict = dict(r)
        related.append({'doc': r_dict, 'relation_type': r_dict['relation_type'], 'direction': 'incoming'})
    workflow = query_db('''
        SELECT wf.*, wt.name as template_name, wt.steps as template_steps, wt.steps_config as template_steps_config
        FROM workflow_instances wf
        JOIN workflow_templates wt ON wf.template_id = wt.id
        WHERE wf.document_id = ?
    ''', (doc_id,), one=True)
    if workflow:
        workflow = dict(workflow)
        tasks = query_db('SELECT * FROM approval_tasks WHERE workflow_instance_id = ? ORDER BY step_index', (workflow['id'],))
        workflow['tasks'] = [dict(t) for t in tasks]
        if workflow['template_steps_config']:
            config = json.loads(workflow['template_steps_config'])
            workflow['steps_detail'] = config.get('steps', [])
        else:
            steps = json.loads(workflow['template_steps'])
            workflow['steps_detail'] = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(steps)]
    else:
        workflow = None
    all_templates = query_db('SELECT * FROM workflow_templates')
    all_templates = [dict(t) for t in all_templates]
    for t in all_templates:
        if t['steps_config']:
            config = json.loads(t['steps_config'])
            t['steps'] = config.get('steps', [])
        else:
            t['steps'] = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(json.loads(t['steps']))]
    return render_template('document.html', document=doc, related=related,
                           workflow=workflow, all_templates=all_templates)

# ---------- Редактирование документа (WYSIWYG) ----------
@app.route('/edit/<int:doc_id>', methods=['GET', 'POST'])
@login_required
def edit_document(doc_id):
    if not has_permission('documents', 'UPDATE'):
        flash('У вас нет прав на редактирование документов', 'danger')
        return redirect(url_for('document_view', doc_id=doc_id))
    doc = query_db('SELECT * FROM documents WHERE id = ?', (doc_id,), one=True)
    if not doc:
        return "Документ не найден", 404
    doc = dict(doc)
    if request.method == 'POST':
        html_content = request.form.get('content', '')
        allowed_tags = ['b', 'strong', 'i', 'em', 'u', 'ul', 'ol', 'li', 'p', 'br', 'span', 'div', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6']
        allowed_attrs = {'*': ['style']}
        clean_html = bleach.clean(html_content, tags=allowed_tags, attributes=allowed_attrs, strip=False)
        execute_db('UPDATE documents SET content = ?, updated_at = CURRENT_TIMESTAMP WHERE id = ?',
                   (clean_html, doc_id))
        flash('Документ обновлён', 'success')
        return redirect(url_for('document_view', doc_id=doc_id))
    return render_template('edit.html', document=doc)

# ---------- Загрузка файла ----------
@app.route('/upload', methods=['GET', 'POST'])
@login_required
def upload_file():
    if not has_permission('documents', 'INSERT'):
        flash('У вас нет прав на загрузку документов', 'danger')
        return redirect(url_for('dashboard'))
    if request.method == 'POST':
        if 'file' not in request.files:
            flash('Файл не выбран', 'danger')
            return redirect(request.url)
        file = request.files['file']
        if file.filename == '':
            flash('Файл не выбран', 'danger')
            return redirect(request.url)
        if file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
            file.save(filepath)

            ext = filename.rsplit('.', 1)[1].lower()
            if ext == 'txt':
                with open(filepath, 'r', encoding='utf-8') as f:
                    text = f.read()
                text = bleach.clean(text, tags=[], strip=True)
                text = text.replace('\n', '<br>')
            elif ext == 'docx':
                text = extract_text_from_docx(filepath)
                allowed_tags = ['p', 'br', 'strong', 'b', 'em', 'i', 'u', 'ul', 'ol', 'li', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6']
                allowed_attrs = {'*': ['style']}
                text = bleach.clean(text, tags=allowed_tags, attributes=allowed_attrs, strip=False)
            else:
                flash('Неподдерживаемый формат файла', 'danger')
                return redirect(request.url)

            doc_id = execute_db(
                'INSERT INTO documents (title, content, source_system, external_id) VALUES (?,?,?,?)',
                (filename, text, 'upload', None)
            )
            storage_id = get_storage_doc_id()
            execute_db(
                'INSERT INTO relations (from_doc_id, to_doc_id, relation_type) VALUES (?,?,?)',
                (doc_id, storage_id, 'uploaded_to')
            )
            flash('Файл успешно загружен и обработан', 'success')
            return redirect(url_for('document_view', doc_id=doc_id))
        else:
            flash('Допустимы только файлы .txt или .docx', 'danger')
            return redirect(request.url)
    return render_template('upload.html')

# ---------- Управление шаблонами согласования ----------
@app.route('/workflows')
@login_required
def workflows_list():
    if not has_permission('workflow_templates', 'SELECT'):
        flash('Нет прав на просмотр шаблонов', 'danger')
        return redirect(url_for('dashboard'))
    templates = query_db('SELECT * FROM workflow_templates')
    templates = [dict(t) for t in templates]
    for t in templates:
        if t['steps_config']:
            config = json.loads(t['steps_config'])
            t['steps'] = config.get('steps', [])
        else:
            t['steps'] = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(json.loads(t['steps']))]
    return render_template('workflows.html', templates=templates)

@app.route('/workflow/new', methods=['GET', 'POST'])
@login_required
def workflow_edit():
    if not has_permission('workflow_templates', 'INSERT'):
        flash('Нет прав на создание шаблонов', 'danger')
        return redirect(url_for('workflows_list'))
    if request.method == 'POST':
        name = request.form['name']
        description = request.form['description']
        steps = []
        steps_config = {'steps': []}
        step_indices = []
        for key in request.form:
            if key.startswith('step_name_'):
                try:
                    idx = int(key.split('_')[-1])
                    step_indices.append(idx)
                except:
                    pass
        step_indices.sort()
        for idx in step_indices:
            step_name = request.form.get(f'step_name_{idx}', '')
            step_role = request.form.get(f'step_role_{idx}', '')
            if not step_name or not step_role:
                continue
            step_deadline = request.form.get(f'step_deadline_{idx}', '')
            step_can_return = request.form.get(f'step_can_return_{idx}', '') == 'on'
            steps.append(step_role)
            steps_config['steps'].append({
                'name': step_name,
                'role': step_role,
                'deadline_hours': int(step_deadline) if step_deadline else None,
                'can_return': step_can_return
            })
        if not steps:
            flash('Добавьте хотя бы один шаг', 'danger')
            return redirect(request.url)
        steps_json = json.dumps(steps)
        steps_config_json = json.dumps(steps_config)
        execute_db('INSERT INTO workflow_templates (name, description, steps, steps_config) VALUES (?,?,?,?)',
                   (name, description, steps_json, steps_config_json))
        flash('Шаблон создан', 'success')
        return redirect(url_for('workflows_list'))
    return render_template('workflow_edit.html', template=None, documents=[])

@app.route('/workflow/<int:template_id>/edit', methods=['GET', 'POST'])
@login_required
def workflow_edit_id(template_id):
    if not has_permission('workflow_templates', 'UPDATE'):
        flash('Нет прав на редактирование шаблонов', 'danger')
        return redirect(url_for('workflows_list'))
    if request.method == 'POST':
        name = request.form['name']
        description = request.form['description']
        steps = []
        steps_config = {'steps': []}
        step_indices = []
        for key in request.form:
            if key.startswith('step_name_'):
                try:
                    idx = int(key.split('_')[-1])
                    step_indices.append(idx)
                except:
                    pass
        step_indices.sort()
        for idx in step_indices:
            step_name = request.form.get(f'step_name_{idx}', '')
            step_role = request.form.get(f'step_role_{idx}', '')
            if not step_name or not step_role:
                continue
            step_deadline = request.form.get(f'step_deadline_{idx}', '')
            step_can_return = request.form.get(f'step_can_return_{idx}', '') == 'on'
            steps.append(step_role)
            steps_config['steps'].append({
                'name': step_name,
                'role': step_role,
                'deadline_hours': int(step_deadline) if step_deadline else None,
                'can_return': step_can_return
            })
        if not steps:
            flash('Добавьте хотя бы один шаг', 'danger')
            return redirect(request.url)
        steps_json = json.dumps(steps)
        steps_config_json = json.dumps(steps_config)
        execute_db('UPDATE workflow_templates SET name=?, description=?, steps=?, steps_config=? WHERE id=?',
                   (name, description, steps_json, steps_config_json, template_id))
        flash('Шаблон обновлён', 'success')
        return redirect(url_for('workflows_list'))
    else:
        template = query_db('SELECT * FROM workflow_templates WHERE id = ?', (template_id,), one=True)
        if not template:
            return "Шаблон не найден", 404
        template = dict(template)
        if template['steps_config']:
            config = json.loads(template['steps_config'])
            template['steps'] = config.get('steps', [])
        else:
            roles = json.loads(template['steps'])
            template['steps'] = [{'name': f'Шаг {i+1}', 'role': r, 'deadline_hours': None, 'can_return': False} for i, r in enumerate(roles)]
        
        # Получаем список документов для выпадающего списка
        documents = query_db('SELECT id, title FROM documents ORDER BY title')
        return render_template('workflow_edit.html', template=template, documents=documents)

# ---------- Запуск процесса из шаблона ----------
@app.route('/workflow/<int:template_id>/start_for_document', methods=['POST'])
@login_required
def start_workflow_from_template(template_id):
    """Запустить процесс согласования для выбранного документа на основе шаблона."""
    if not has_permission('workflow_instances', 'INSERT'):
        flash('Нет прав на запуск процессов', 'danger')
        return redirect(url_for('workflow_edit_id', template_id=template_id))

    document_id = request.form.get('document_id')
    if not document_id:
        flash('Не выбран документ', 'danger')
        return redirect(url_for('workflow_edit_id', template_id=template_id))

    # Проверяем, нет ли уже активного процесса для этого документа
    existing = query_db('SELECT id FROM workflow_instances WHERE document_id = ? AND status = "active"', (document_id,), one=True)
    if existing:
        flash('Для этого документа уже есть активный процесс согласования', 'warning')
        return redirect(url_for('workflow_edit_id', template_id=template_id))

    template = query_db('SELECT * FROM workflow_templates WHERE id = ?', (template_id,), one=True)
    if not template:
        flash('Шаблон не найден', 'danger')
        return redirect(url_for('workflow_edit_id', template_id=template_id))
    template = dict(template)

    wf_id = execute_db(
        'INSERT INTO workflow_instances (document_id, template_id, status, current_step) VALUES (?,?,?,?)',
        (document_id, template_id, 'active', 0)
    )

    if template['steps_config']:
        config = json.loads(template['steps_config'])
        steps_detail = config.get('steps', [])
    else:
        roles = json.loads(template['steps'])
        steps_detail = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(roles)]

    first_step = steps_detail[0]
    deadline = None
    if first_step.get('deadline_hours'):
        deadline = datetime.now() + timedelta(hours=first_step['deadline_hours'])

    execute_db(
        '''INSERT INTO approval_tasks 
           (workflow_instance_id, step_index, assigned_role, status, deadline, can_return) 
           VALUES (?,?,?,?,?,?)''',
        (wf_id, 0, first_step['role'], 'pending', deadline, 1 if first_step.get('can_return') else 0)
    )

    flash(f'Процесс согласования запущен для документа {document_id} (ID процесса {wf_id})', 'success')
    return redirect(url_for('document_view', doc_id=document_id))

# ---------- Массовое управление процессами шаблона ----------
@app.route('/workflow/<int:template_id>/complete_all', methods=['POST'])
@login_required
def complete_all_workflows(template_id):
    """Завершить все активные процессы для шаблона."""
    if g.user['role'] not in ['admin', 'manager']:
        flash('Недостаточно прав для массового завершения процессов', 'danger')
        return redirect(url_for('workflow_edit_id', template_id=template_id))
    
    processes = query_db('SELECT id FROM workflow_instances WHERE template_id = ? AND status = "active"', (template_id,))
    count = len(processes)
    if count == 0:
        flash('Нет активных процессов для этого шаблона', 'info')
        return redirect(url_for('workflow_edit_id', template_id=template_id))
    
    for proc in processes:
        execute_db('UPDATE workflow_instances SET status="completed" WHERE id=?', (proc['id'],))
        execute_db('UPDATE approval_tasks SET status="cancelled", updated_at=CURRENT_TIMESTAMP WHERE workflow_instance_id=? AND status="pending"', (proc['id'],))
    
    flash(f'Успешно завершено {count} активных процессов', 'success')
    return redirect(url_for('workflow_edit_id', template_id=template_id))

@app.route('/workflow/<int:template_id>/reject_all', methods=['POST'])
@login_required
def reject_all_workflows(template_id):
    """Отклонить все активные процессы для шаблона."""
    if g.user['role'] not in ['admin', 'manager']:
        flash('Недостаточно прав для массового отклонения процессов', 'danger')
        return redirect(url_for('workflow_edit_id', template_id=template_id))
    
    processes = query_db('SELECT id FROM workflow_instances WHERE template_id = ? AND status = "active"', (template_id,))
    count = len(processes)
    if count == 0:
        flash('Нет активных процессов для этого шаблона', 'info')
        return redirect(url_for('workflow_edit_id', template_id=template_id))
    
    for proc in processes:
        execute_db('UPDATE workflow_instances SET status="rejected" WHERE id=?', (proc['id'],))
        execute_db('UPDATE approval_tasks SET status="cancelled", updated_at=CURRENT_TIMESTAMP WHERE workflow_instance_id=? AND status="pending"', (proc['id'],))
    
    flash(f'Успешно отклонено {count} активных процессов', 'danger')
    return redirect(url_for('workflow_edit_id', template_id=template_id))

# ---------- Запуск процесса из документа (существующий) ----------
@app.route('/document/<int:doc_id>/start_workflow', methods=['POST'])
@login_required
def start_workflow(doc_id):
    if not has_permission('workflow_instances', 'INSERT'):
        flash('Нет прав на запуск процессов', 'danger')
        return redirect(url_for('document_view', doc_id=doc_id))
    template_id = request.form['template_id']
    existing = query_db('SELECT id FROM workflow_instances WHERE document_id = ? AND status = "active"', (doc_id,), one=True)
    if existing:
        flash('Для этого документа уже есть активный процесс согласования', 'warning')
        return redirect(url_for('document_view', doc_id=doc_id))
    template = query_db('SELECT * FROM workflow_templates WHERE id = ?', (template_id,), one=True)
    if not template:
        flash('Шаблон не найден', 'danger')
        return redirect(url_for('document_view', doc_id=doc_id))
    template = dict(template)
    wf_id = execute_db(
        'INSERT INTO workflow_instances (document_id, template_id, status, current_step) VALUES (?,?,?,?)',
        (doc_id, template_id, 'active', 0)
    )

    if template['steps_config']:
        config = json.loads(template['steps_config'])
        steps_detail = config.get('steps', [])
    else:
        roles = json.loads(template['steps'])
        steps_detail = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(roles)]

    first_step = steps_detail[0]
    deadline = None
    if first_step.get('deadline_hours'):
        deadline = datetime.now() + timedelta(hours=first_step['deadline_hours'])
    execute_db(
        '''INSERT INTO approval_tasks 
           (workflow_instance_id, step_index, assigned_role, status, deadline, can_return) 
           VALUES (?,?,?,?,?,?)''',
        (wf_id, 0, first_step['role'], 'pending', deadline, 1 if first_step.get('can_return') else 0)
    )
    flash('Процесс согласования запущен', 'success')
    return redirect(url_for('document_view', doc_id=doc_id))

# ---------- Действия над процессом (завершить, отклонить) ----------
@app.route('/process/<int:process_id>/complete', methods=['POST'])
@login_required
def complete_process(process_id):
    workflow = query_db('SELECT * FROM workflow_instances WHERE id = ?', (process_id,), one=True)
    if not workflow:
        flash('Процесс не найден', 'danger')
        return redirect(url_for('dashboard'))
    if workflow['status'] != 'active':
        flash('Процесс уже завершён или отклонён', 'warning')
        return redirect(url_for('document_view', doc_id=workflow['document_id']))
    # Завершаем процесс
    execute_db('UPDATE workflow_instances SET status="completed" WHERE id=?', (process_id,))
    # Все незавершённые задачи этого процесса помечаем как отменённые (опционально)
    execute_db('UPDATE approval_tasks SET status="cancelled", updated_at=CURRENT_TIMESTAMP WHERE workflow_instance_id=? AND status="pending"', (process_id,))
    flash('Процесс успешно завершён', 'success')
    return redirect(url_for('document_view', doc_id=workflow['document_id']))

@app.route('/process/<int:process_id>/reject', methods=['POST'])
@login_required
def reject_process(process_id):
    workflow = query_db('SELECT * FROM workflow_instances WHERE id = ?', (process_id,), one=True)
    if not workflow:
        flash('Процесс не найден', 'danger')
        return redirect(url_for('dashboard'))
    if workflow['status'] != 'active':
        flash('Процесс уже завершён или отклонён', 'warning')
        return redirect(url_for('document_view', doc_id=workflow['document_id']))
    # Отклоняем процесс
    execute_db('UPDATE workflow_instances SET status="rejected" WHERE id=?', (process_id,))
    # Все незавершённые задачи этого процесса помечаем как отменённые
    execute_db('UPDATE approval_tasks SET status="cancelled", updated_at=CURRENT_TIMESTAMP WHERE workflow_instance_id=? AND status="pending"', (process_id,))
    flash('Процесс отклонён', 'danger')
    return redirect(url_for('document_view', doc_id=workflow['document_id']))

# ---------- Просмотр и выполнение задачи согласования ----------
@app.route('/task/<int:task_id>', methods=['GET', 'POST'])
@login_required
def approval_task_view(task_id):
    task = query_db('SELECT * FROM approval_tasks WHERE id = ?', (task_id,), one=True)
    if not task:
        return "Задача не найдена", 404
    task = dict(task)
    workflow = query_db('SELECT * FROM workflow_instances WHERE id = ?', (task['workflow_instance_id'],), one=True)
    workflow = dict(workflow)
    document = query_db('SELECT * FROM documents WHERE id = ?', (workflow['document_id'],), one=True)
    document = dict(document)
    template = query_db('SELECT * FROM workflow_templates WHERE id = ?', (workflow['template_id'],), one=True)
    template = dict(template)
    if template['steps_config']:
        config = json.loads(template['steps_config'])
        steps_detail = config.get('steps', [])
    else:
        roles = json.loads(template['steps'])
        steps_detail = [{'role': r, 'name': f'Шаг {i+1}', 'deadline_hours': None, 'can_return': False} for i, r in enumerate(roles)]
    if g.user['role'] != task['assigned_role'] and g.user['role'] != 'admin':
        flash('У вас нет прав на выполнение этой задачи', 'danger')
        return redirect(url_for('dashboard'))
    if request.method == 'POST':
        action = request.form['action']
        comments = request.form.get('comments', '')
        if action == 'approve':
            new_status = 'approved'
            execute_db('UPDATE approval_tasks SET status=?, comments=?, updated_at = CURRENT_TIMESTAMP WHERE id=?',
                       (new_status, comments, task_id))
            current_step = workflow['current_step']
            if current_step + 1 < len(steps_detail):
                next_step = current_step + 1
                next_step_detail = steps_detail[next_step]
                deadline = None
                if next_step_detail.get('deadline_hours'):
                    deadline = datetime.now() + timedelta(hours=next_step_detail['deadline_hours'])
                execute_db(
                    '''INSERT INTO approval_tasks 
                       (workflow_instance_id, step_index, assigned_role, status, deadline, can_return) 
                       VALUES (?,?,?,?,?,?)''',
                    (workflow['id'], next_step, next_step_detail['role'], 'pending',
                     deadline, 1 if next_step_detail.get('can_return') else 0)
                )
                execute_db('UPDATE workflow_instances SET current_step=? WHERE id=?', (next_step, workflow['id']))
                flash('Шаг одобрен, создана следующая задача', 'success')
            else:
                execute_db('UPDATE workflow_instances SET status="completed" WHERE id=?', (workflow['id'],))
                flash('Документ полностью согласован', 'success')
        elif action == 'return':
            if not task['can_return']:
                flash('Возврат на доработку для этой задачи не разрешён', 'danger')
                return redirect(url_for('approval_task_view', task_id=task_id))
            current_step = workflow['current_step']
            if current_step == 0:
                execute_db('UPDATE workflow_instances SET status="returned" WHERE id=?', (workflow['id'],))
                flash('Процесс возвращён на доработку. Документ может быть отредактирован.', 'warning')
            else:
                prev_step = current_step - 1
                prev_step_detail = steps_detail[prev_step]
                deadline = None
                if prev_step_detail.get('deadline_hours'):
                    deadline = datetime.now() + timedelta(hours=prev_step_detail['deadline_hours'])
                new_task_id = execute_db(
                    '''INSERT INTO approval_tasks 
                       (workflow_instance_id, step_index, assigned_role, status, deadline, can_return, parent_task_id) 
                       VALUES (?,?,?,?,?,?,?)''',
                    (workflow['id'], prev_step, prev_step_detail['role'], 'pending',
                     deadline, 1 if prev_step_detail.get('can_return') else 0, task_id)
                )
                execute_db('UPDATE approval_tasks SET status="returned", comments=?, updated_at=CURRENT_TIMESTAMP WHERE id=?',
                           (comments, task_id))
                execute_db('UPDATE workflow_instances SET current_step=? WHERE id=?', (prev_step, workflow['id']))
                flash('Задача возвращена на предыдущий шаг', 'info')
        else:  # reject
            execute_db('UPDATE approval_tasks SET status="rejected", comments=?, updated_at=CURRENT_TIMESTAMP WHERE id=?',
                       (comments, task_id))
            execute_db('UPDATE workflow_instances SET status="rejected" WHERE id=?', (workflow['id'],))
            flash('Документ отклонён', 'danger')
        return redirect(url_for('dashboard'))
    return render_template('approval_task.html', task=task, document=document, workflow=workflow, steps_detail=steps_detail)

# ---------- Аудит ----------
@app.route('/audit')
@login_required
def audit_view():
    if g.user['role'] != 'admin':
        flash('Только администратор может просматривать аудит', 'danger')
        return redirect(url_for('dashboard'))
    logs = query_db('SELECT * FROM audit_log ORDER BY changed_at DESC LIMIT 200')
    logs = [dict(l) for l in logs]
    return render_template('audit.html', logs=logs)

# ---------- Резервное копирование ----------
def backup_database():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_file = os.path.join(BACKUP_DIR, f'edoc_backup_{timestamp}.db')
    shutil.copy2(DATABASE, backup_file)
    return backup_file

def list_backups():
    files = [f for f in os.listdir(BACKUP_DIR) if f.endswith('.db')]
    files.sort(reverse=True)
    return files

def restore_database(backup_filename):
    backup_path = os.path.join(BACKUP_DIR, backup_filename)
    if os.path.exists(backup_path):
        shutil.copy2(backup_path, DATABASE)
        return True
    return False

@app.route('/backup', methods=['GET', 'POST'])
@login_required
def backup_page():
    if g.user['role'] != 'admin':
        flash('Только администратор может управлять резервными копиями', 'danger')
        return redirect(url_for('dashboard'))
    if request.method == 'POST':
        action = request.form.get('action')
        if action == 'create':
            try:
                file = backup_database()
                flash(f'Резервная копия создана: {os.path.basename(file)}', 'success')
            except Exception as e:
                flash(f'Ошибка при создании копии: {e}', 'danger')
        elif action == 'restore':
            filename = request.form.get('backup_file')
            if filename:
                try:
                    restore_database(filename)
                    flash('Восстановление выполнено. Пожалуйста, перезайдите в систему.', 'success')
                    return redirect(url_for('logout'))
                except Exception as e:
                    flash(f'Ошибка восстановления: {e}', 'danger')
    backups = list_backups()
    return render_template('backup.html', backups=backups)

# ---------- Политика безопасности ----------
@app.route('/policy')
@login_required
def policy():
    policy_text = """
    <h1>Политика безопасности системы документооборота EDoC</h1>
    <p><strong>Версия:</strong> 1.0</p>
    <h2>1. Роли и привилегии</h2>
    <table border="1" cellpadding="5">
        <tr><th>Роль</th><th>Таблицы</th><th>SELECT</th><th>INSERT</th><th>UPDATE</th><th>DELETE</th></tr>
        <tr><td>admin</td><td>все</td><td>✔</td><td>✔</td><td>✔</td><td>✔</td></tr>
        <tr><td>manager</td><td>documents</td><td>✔</td><td>✔</td><td>✔</td><td>✘</td></tr>
        <tr><td>manager</td><td>workflow_templates, workflow_instances</td><td>✔</td><td>✔</td><td>✔</td><td>✘</td> </tr>
        <tr><td>manager</td><td>approval_tasks</td><td>✔</td><td>✘</td><td>✔</td><td>✘</td> </tr>
        <tr><td>developer</td><td>documents, workflow_templates, workflow_instances</td><td>✔</td><td>✘</td><td>✘</td><td>✘</td> </tr>
        <tr><td>developer</td><td>approval_tasks (только свои)</td><td>✔</td><td>✘</td><td>✔</td><td>✘</td> </tr>
        <tr><td>lawyer</td><td>documents, workflow_templates, workflow_instances</td><td>✔</td><td>✘</td><td>✘</td><td>✘</td> </tr>
        <tr><td>lawyer</td><td>approval_tasks (только свои)</td><td>✔</td><td>✘</td><td>✔</td><td>✘</td> </tr>
    </table>
    <h2>2. Аутентификация</h2>
    <p>Пароли хранятся в виде хеша SHA-256. Сессии управляются через Flask.</p>
    <h2>3. Аудит</h2>
    <p>Все изменения в таблицах documents, workflow_instances, workflow_templates, approval_tasks фиксируются в audit_log. Изменения статуса задач дополнительно сохраняются в task_status_history.</p>
    <h2>4. Целостность данных</h2>
    <ul>
        <li>Внешние ключи с каскадным удалением.</li>
        <li>Автоматическое обновление поля updated_at при изменениях (выполняется в коде приложения).</li>
        <li>Ограничение: сотрудник может выполнять только задачи, назначенные на его роль.</li>
    </ul>
    <h2>5. Резервное копирование</h2>
    <p>Резервные копии создаются в папке backups/. Восстановление доступно только администратору.</p>
    <h2>6. Ответственный</h2>
    <p>Администратор БД: (alex_admin).</p>
    """
    return render_template('policy.html', policy_text=policy_text)

# ---------- Тестирование ----------
@app.route('/test', methods=['GET', 'POST'])
@login_required
def test_page():
    if g.user['role'] != 'admin':
        flash('Только администратор может запускать тесты', 'danger')
        return redirect(url_for('dashboard'))
    test_output = ""
    if request.method == 'POST':
        import io
        import sys
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        try:
            print("=== ТЕСТИРОВАНИЕ СИСТЕМЫ БЕЗОПАСНОСТИ ===")
            print("1. Проверка прав администратора...")
            if has_permission('documents', 'DELETE'):
                print("   ✓ Администратор может удалять документы")
            else:
                print("   ✗ Ошибка: администратор не может удалять")
            print("\n2. Проверка прав менеджера (имитация)...")
            print("   (тест пропущен, т.к. требует смены пользователя)")
            print("\n3. Проверка триггера аудита...")
            before = query_db('SELECT COUNT(*) as cnt FROM audit_log', one=True)['cnt']
            execute_db("INSERT INTO documents (title, content) VALUES (?,?)", ('Тестовый документ', 'test'))
            after = query_db('SELECT COUNT(*) as cnt FROM audit_log', one=True)['cnt']
            if after > before:
                print("   ✓ Аудит работает (запись добавлена)")
            else:
                print("   ✗ Аудит не сработал")
            print("\n4. Проверка резервного копирования...")
            try:
                backup_database()
                backups = list_backups()
                if backups:
                    print(f"   ✓ Резервная копия создана. Всего копий: {len(backups)}")
                else:
                    print("   ✗ Не удалось создать копию")
            except Exception as e:
                print(f"   ✗ Ошибка: {e}")
            print("\n=== ТЕСТИРОВАНИЕ ЗАВЕРШЕНО ===")
        except Exception as e:
            print(f"Ошибка выполнения тестов: {e}")
        test_output = sys.stdout.getvalue()
        sys.stdout = old_stdout
    return render_template('test.html', test_output=test_output)

# ---------- Запуск ----------
if __name__ == '__main__':
    if not os.path.exists(DATABASE):
        with app.app_context():
            init_db()
    from werkzeug.serving import run_simple
    run_simple('localhost', 5000, app, use_reloader=False)